# Finish Time Model

**Gender-Specific Power Law with Course Effects**

This model estimates ultrarunning race pace using gender-specific power law relationships between distance and pace, accounting for course-specific difficulty. The model pools data across multiple years of the same race course to improve parameter estimation.

**Model Data:** All finishers from races that report DNFs

**Data Subsetting:** Well connected k-core subset + closure courses/participants up to `max_entities`

**Hyperparameters**

- `course_finish_time_multiplier_std`: Variation in course difficulty (finish times)

**Runner Parameters**

- `pace_distance_effect[gender]`: Distance exponent (how pace degrades with distance, per gender)
- `finish_time_noise[gender]`: Observation noise (gender-specific)
- `pace_marathon[gender]`: Average pace at marathon distance (per gender)

**Course Parameters (n_courses)**

- `course_finish_time_multiplier`: Course-specific pace adjustment (shared across genders)

## Setup

Load and process the whole dataset

In [ ]:
# libraries
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import pymc as pm
import numpy as np
import arviz as az
from pathlib import Path
from plotly.subplots import make_subplots
from matplotlib import pyplot as plt
from IPython.display import display, Image
from scipy.stats import pearsonr
import time

# local functions
from utils.data_processing import load_results, process_results, filter_races_with_dnfs
from utils.kcore_subsetting import subset_kcore_data
from utils.mcmc_notifications import notify_mcmc_start, notify_mcmc_complete, notify_mcmc_error
from utils.empirical_priors import (
    calculate_gender_marathon_pace,
    calculate_gender_distance_exponent,
    calculate_course_variation
)
from utils.utils import plot_posterior_diagnostics

In [ ]:
# define variables

# Standard race distances for analysis and visualization
# Format: (distance_miles, label, tolerance_for_binning)
standard_distances = [
    (6.21371, '10K', 0.3),
    (10.0, '10mi', 0.3),
    (13.1, 'Half', 0.3),
    (26.2, 'Marathon', 0.5),
    (31.0686, '50K', 0.5),
    (50.0, '50mi', 1.0),
    (62.1371, '100K', 1.0),
    (100.0, '100mi', 1.0)
]

reference_distance = 26.2  # Marathon

# K-core parameters chosen from kcore_optimization.ipynb Pareto frontier
# TODO: Update these values based on your Pareto frontier analysis
alpha = 5  # Minimum courses per k-core runner
beta = 25  # Minimum runners per k-core course

tune = 1000
draws = 2000
target_accept = 0.9

model_dir = Path('../data/cache/model_1')
os.makedirs(model_dir, exist_ok=True)

subset_dir = f'{model_dir}/alpha{alpha}_beta{beta}'
os.makedirs(subset_dir, exist_ok=True)

In [3]:
# load and process data

results = load_results()
results = process_results(results)

KeyboardInterrupt: 

### Data Filtering

Filter the dataset to the results relevant to this model

In [ ]:
# this model does not attempt to model DNF reporting or DNFs themselves
results = filter_races_with_dnfs(results)
results = results[results.finished == 1]

### Data Subsetting

Subset the data to a well-connected subgraph for computational efficiency 

In [ ]:
# Apply k-core flagging with course-completion closure
from utils.kcore_subsetting import subset_kcore_data
results = subset_kcore_data(results, alpha=alpha, beta=beta)
model_data = results[results['in_kcore'] | results['in_closure']]

print(f"\nK-core subset (α={alpha}, β={beta}):")
print(f"  K-core results: {results['in_kcore'].sum():,}")
print(f"  Closure results: {results['in_closure'].sum():,}")
print(f"  Total for modeling: {len(model_data):,}")
print(f"  K-core courses: {results[results['in_kcore']]['name'].nunique():,}")
print(f"  K-core runners: {results[results['in_kcore']]['participant_id'].nunique():,}")

## Specification

Define the model

### Empirical Priors

Compute empirical priors on all the results relevant to this model

In [ ]:
# Calculate empirical priors using simple, focused functions
# Each function computes ONE specific prior from the data

# Gender-specific marathon pace (mu_pace prior)
mu_pace_m = calculate_gender_marathon_pace(results, 'M', reference_distance)
mu_pace_f = calculate_gender_marathon_pace(results, 'F', reference_distance)

# Gender-specific distance exponent (beta prior)
beta_m = calculate_gender_distance_exponent(results, 'M', reference_distance)
beta_f = calculate_gender_distance_exponent(results, 'F', reference_distance)

# Course-level variation (sigma_course prior)
sigma_course_prior = calculate_course_variation(results, reference_distance)

### Generative Model

In [ ]:
# create index mappings
unique_courses = model_data['name'].unique()
course_to_idx = {course: idx for idx, course in enumerate(unique_courses)}
model_data['course_idx'] = model_data['name'].map(course_to_idx)

# Create gender index mapping
unique_genders = ['M', 'F']  # Explicit ordering
gender_to_idx = {'M': 0, 'F': 1}
model_data['gender_idx'] = model_data['gender'].map(gender_to_idx)

# Extract arrays for PyMC model
n_courses = len(unique_courses)
n_genders = len(unique_genders)
n_observations = len(model_data)

# Create arrays for the model
course_indices = model_data['course_idx'].values
gender_indices = model_data['gender_idx'].values
race_distances = model_data['distance_miles'].values
observed_times = model_data['time_ms'].values / 1000  # Convert milliseconds to seconds

# Power law reference distance (marathon distance in miles)
reference_distance = 26.2

# ============================================================================
# GENERATIVE MODEL
# ============================================================================
# This is a hierarchical Bayesian model for ultramarathon finish times
#
# Key features:
# - Gender-specific power law relationship between pace and distance
# - Course-level random effects shared across genders
# - Course effects persist across years and are shared across genders
# - Distance scaling varies by gender
#
# Uses non-centered parameterization to avoid funnel geometry

coords = {
    'course': unique_courses,
    'gender': unique_genders,
    'finishers': range(n_observations),
}

with pm.Model(coords=coords) as model:
    
    # --- FINISH TIME MODEL: Population Hyperpriors (gender-specific) ---
    # Average pace at marathon distance (per gender)
    pace_marathon = pm.Normal(
        'pace_marathon', 
        mu=[mu_pace_m, mu_pace_f],
        sigma=0.1,
        dims='gender'
    )
    
    # Distance exponent (how much pace degrades with distance, per gender)
    pace_distance_effect = pm.Normal(
        'pace_distance_effect', 
        mu=[beta_m, beta_f],
        sigma=0.05,
        dims='gender'
    )
    
    # Course difficulty variation - shared across genders
    course_finish_time_multiplier_std = pm.HalfNormal(
        'course_finish_time_multiplier_std', 
        sigma=sigma_course_prior
    )
    
    # Observation noise - gender-specific
    finish_time_noise = pm.HalfNormal(
        'finish_time_noise', 
        sigma=0.15,
        dims='gender'
    )
    
    # --- FINISH TIME MODEL: Course-level effects (non-centered parameterization) ---
    course_finish_time_multiplier_raw = pm.Normal(
        'course_finish_time_multiplier_raw', 
        mu=0, 
        sigma=1, 
        dims='course'
    )
    course_finish_time_multiplier = pm.Deterministic(
        'course_finish_time_multiplier', 
        course_finish_time_multiplier_std * course_finish_time_multiplier_raw, 
        dims='course'
    )
    
    # --- FINISH TIME MODEL: Likelihood ---
    # Power law: log(pace) = baseline[gender] + pace_distance_effect[gender] * log(distance/reference)
    log_distance_ratio = pm.math.log(race_distances / reference_distance)
    
    # Expected log pace for each observation (index by gender)
    expected_log_pace = (
        pace_marathon[gender_indices] +
        pace_distance_effect[gender_indices] * log_distance_ratio +
        course_finish_time_multiplier[course_indices]
    )
    
    # Convert back to linear scale (pace in sec/mile)
    expected_pace = pm.Deterministic('expected_pace', pm.math.exp(expected_log_pace), dims='finishers')
    
    # Convert pace to total time (seconds)
    expected_time = expected_pace * race_distances
    
    # Likelihood with gender-specific observation noise
    pm.Normal(
        'finish_times', 
        mu=expected_time, 
        sigma=finish_time_noise[gender_indices] * race_distances, 
        observed=observed_times,
        dims='finishers'
    )

# Display model structure as graphviz diagram
graph = pm.model_to_graphviz(model)

# Save to model directory with smaller size
graph_file = f'{model_dir}/model_structure'
graph.graph_attr['dpi'] = '72'  # Reduce DPI for smaller file size
graph.render(graph_file, format='png', cleanup=True)
print(f"Saved model structure to {graph_file}.png")

# Display in notebook
graph

In [ ]:
with model:
    prior_pred = pm.sample_prior_predictive(samples=10000, random_seed=37)

### Prior Visualization

In [ ]:
# Prior Distribution Visualizations
# Extract prior samples from prior_pred and plot as KDEs to show theoretical distributions
pace_marathon_prior = prior_pred.prior["pace_marathon"].values  # Shape: (chains, draws, n_genders)
pace_distance_effect_prior = prior_pred.prior["pace_distance_effect"].values
course_finish_time_multiplier_std_prior = prior_pred.prior["course_finish_time_multiplier_std"].values
finish_time_noise_prior = prior_pred.prior["finish_time_noise"].values

# Flatten to (samples,) for scalar parameters or (samples, dim) for vector parameters
pace_marathon_flat = pace_marathon_prior.reshape(-1, n_genders)
pace_distance_effect_flat = pace_distance_effect_prior.reshape(-1, n_genders)
course_finish_time_multiplier_std_flat = course_finish_time_multiplier_std_prior.flatten()
finish_time_noise_flat = finish_time_noise_prior.reshape(-1, n_genders)

# Convert to interpretable units
# pace_marathon: exp(log pace) = pace in min/mile
pace_marathon_interpretable = np.exp(pace_marathon_flat)

# pace_distance_effect: already interpretable (unitless exponent)
pace_distance_effect_interpretable = pace_distance_effect_flat

# course_finish_time_multiplier_std: convert to time multiplier via exp(std)
course_mult_std_interpretable = np.exp(course_finish_time_multiplier_std_flat)

# finish_time_noise: convert to time multiplier via exp(std)
finish_time_noise_interpretable = np.exp(finish_time_noise_flat)

# ============================================================================
# PLOT 1: Hyperparameter Priors (only gender and population-level parameters)
# ============================================================================
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

gender_colors_plot = {'M': 'steelblue', 'F': 'coral'}
gender_labels_plot = {'M': 'Male', 'F': 'Female'}

# 1. pace_marathon (gender-specific) - in min/mile
ax = axes[0]
for gender_idx, gender in enumerate(unique_genders):
    # Use seaborn kdeplot for smooth KDE curves
    import seaborn as sns
    sns.kdeplot(data=pace_marathon_interpretable[:, gender_idx], 
               ax=ax, color=gender_colors_plot[gender], 
               label=gender_labels_plot[gender], linewidth=2.5, fill=False)
    
    # Add empirical prior as vertical line
    empirical_value = np.exp(mu_pace_m if gender == 'M' else mu_pace_f)
    ax.axvline(x=empirical_value, color=gender_colors_plot[gender], 
              linestyle='--', linewidth=2, alpha=0.8)

ax.set_xlabel('Pace (min/mile)', fontsize=11, fontweight='bold')
ax.set_ylabel('Density', fontsize=11, fontweight='bold')
ax.set_title('pace_marathon\nAverage running pace at marathon distance', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)

# 2. pace_distance_effect (gender-specific)
ax = axes[1]
for gender_idx, gender in enumerate(unique_genders):
    sns.kdeplot(data=pace_distance_effect_interpretable[:, gender_idx], 
               ax=ax, color=gender_colors_plot[gender], 
               label=gender_labels_plot[gender], linewidth=2.5, fill=False)
    
    # Add empirical prior as vertical line
    empirical_value = beta_m if gender == 'M' else beta_f
    ax.axvline(x=empirical_value, color=gender_colors_plot[gender], 
              linestyle='--', linewidth=2, alpha=0.8)

ax.set_xlabel('Exponent (unitless)', fontsize=11, fontweight='bold')
ax.set_ylabel('Density', fontsize=11, fontweight='bold')
ax.set_title('pace_distance_effect\nHow pace slows as distance doubles', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)

# 3. course_finish_time_multiplier_std (scalar) - as time multiplier
ax = axes[2]
sns.kdeplot(data=course_mult_std_interpretable, ax=ax, 
           color='#4a4a4a', linewidth=2.5, fill=False)

empirical_multiplier = np.exp(sigma_course_prior)
ax.axvline(x=empirical_multiplier, color='#4a4a4a', linestyle='--', 
          linewidth=2, alpha=0.8, label='Empirical prior')
ax.axvline(x=1.0, color='red', linestyle=':', linewidth=1.5, alpha=0.5, 
          label='No variation')

ax.set_xlabel('Time Multiplier (×)', fontsize=11, fontweight='bold')
ax.set_ylabel('Density', fontsize=11, fontweight='bold')
ax.set_title('course_finish_time_multiplier_std\nTypical course-to-course time variation', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)

# 4. finish_time_noise (gender-specific) - as time multiplier
ax = axes[3]
for gender_idx, gender in enumerate(unique_genders):
    sns.kdeplot(data=finish_time_noise_interpretable[:, gender_idx], 
               ax=ax, color=gender_colors_plot[gender], 
               label=gender_labels_plot[gender], linewidth=2.5, fill=False)

ax.axvline(x=1.0, color='red', linestyle=':', linewidth=1.5, alpha=0.5, 
          label='No variation')

ax.set_xlabel('Time Multiplier (×)', fontsize=11, fontweight='bold')
ax.set_ylabel('Density', fontsize=11, fontweight='bold')
ax.set_title('finish_time_noise\nWithin-person variability in finish times', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)

plt.suptitle('Prior Distributions: Hyperparameters', fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()

### Prior Predictive Check

Check that the data generated by the model matches the observed data

In [ ]:
# Prior Predictive Check: Finish Times by Distance
# Extract prior samples
pace_marathon_prior = prior_pred.prior["pace_marathon"].values  # Shape: (chains, draws, n_genders)
pace_distance_effect_prior = prior_pred.prior["pace_distance_effect"].values  # Shape: (chains, draws, n_genders)
course_finish_time_multiplier_prior = prior_pred.prior["course_finish_time_multiplier"].values
baseline_flat = course_finish_time_multiplier_prior.reshape(-1, course_finish_time_multiplier_prior.shape[-1])
finish_time_noise_prior = prior_pred.prior["finish_time_noise"].values.flatten()

# Gender label mapping for display
gender_display_names = {'M': 'Male', 'F': 'Female'}

# Use FULL dataset for observed distributions (not model_data subset)
# Extract data from results DataFrame
full_observed_times = results['time_ms'].values / 60000  # Convert to minutes
full_gender_indices = results['gender'].map(gender_to_idx).values
full_race_distances = results['distance_miles'].values

# Create separate figure for each gender
for gender_idx, gender in enumerate(unique_genders):
    # Create figure with 2 rows x 4 columns for this gender
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    axes = axes.flatten()  # Flatten to 1D array for easier indexing
    
    gender_mask = full_gender_indices == gender_idx
    
    # Iterate over all 8 distances
    for col_idx, distance_tuple in enumerate(standard_distances):
        dist_value, name, tolerance = distance_tuple  # Explicit unpacking with correct order
        ax = axes[col_idx]
        
        # Filter observed data for this distance and gender (from FULL dataset)
        obs_mask = np.abs(full_race_distances - dist_value) < tolerance
        combined_mask = obs_mask & gender_mask
        
        if not combined_mask.any():
            ax.axis('off')
            ax.set_title(f'{name}\nNo data', fontsize=10)
            continue
        
        obs_times_binned = full_observed_times[combined_mask]
        
        # Generate prior predictions for this gender
        n_prior_samples = min(len(obs_times_binned) * 2, 5000)
        random_draws = np.random.choice(pace_marathon_prior.shape[0] * pace_marathon_prior.shape[1], n_prior_samples)
        random_courses = np.random.choice(n_courses, n_prior_samples)
        
        # Flatten gender-specific parameters
        pace_marathon_flat = pace_marathon_prior.reshape(-1, n_genders)[:, gender_idx]
        pace_distance_effect_flat = pace_distance_effect_prior.reshape(-1, n_genders)[:, gender_idx]
        
        # Power law: log(pace) = pace_marathon[gender] + pace_distance_effect[gender] * log(distance/reference) + course_finish_time_multiplier
        log_distance_ratio = np.log(dist_value / reference_distance)
        
        log_prior_pace = (
            pace_marathon_flat[random_draws] +
            pace_distance_effect_flat[random_draws] * log_distance_ratio +
            baseline_flat[random_draws, random_courses]
        )
        
        # Add observation noise
        log_prior_pace_with_noise = np.random.normal(
            log_prior_pace,
            finish_time_noise_prior[random_draws]
        )
        prior_times = np.exp(log_prior_pace_with_noise) * dist_value
        
        # Calculate 99th percentile for x-axis limit (per facet)
        combined_data = np.concatenate([obs_times_binned, prior_times])
        xlim_max = np.percentile(combined_data, 99)
        
        # Plot with step histograms (matching model_2 style)
        # Only include labels for first subplot to create single legend
        if col_idx == 0:
            ax.hist(obs_times_binned, bins='auto', histtype='step', linewidth=2,
                   label='Observed', density=True, color='steelblue')
            ax.hist(prior_times, bins='auto', histtype='step', linewidth=2,
                   label='Prior', density=True, color='orange')
        else:
            ax.hist(obs_times_binned, bins='auto', histtype='step', linewidth=2,
                   density=True, color='steelblue')
            ax.hist(prior_times, bins='auto', histtype='step', linewidth=2,
                   density=True, color='orange')
        
        ax.set_xlabel('Time (min)', fontsize=10)
        ax.set_ylabel('Density', fontsize=10)
        ax.set_title(f'{name} ({dist_value:.1f}mi)\n{len(obs_times_binned):,} finishers', 
                    fontsize=10)
        ax.tick_params(labelsize=9)
        ax.set_xlim(0, xlim_max)  # Set x-axis limit to 99th percentile
    
    # Create single horizontal legend below the title
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 0.98), 
              ncol=2, fontsize=10, frameon=False)
    
    fig.suptitle(f'Prior Predictive Check: {gender_display_names[gender]} Finish Times', 
                 fontsize=14, fontweight='bold', y=1.00)
    plt.tight_layout(rect=[0, 0, 1, 0.96])  # Leave space for legend
    plt.show()


### Geometry Check

In [ ]:
# PRE-SAMPLING GEOMETRY DIAGNOSTIC
# Comprehensive check for sampling issues using ArviZ
# Detects: funnels, high correlations, poor prior specification

# ============================================================================
# GLOBAL FUNNEL CHECK: Hyperprior vs Aggregate Course Variance
# ============================================================================
# Check if the scale parameter (hyperprior) correlates with the variance of ALL raw course parameters
# This is the RIGHT way to check for funnel geometry in hierarchical models

course_finish_time_multiplier_std_prior_samples = prior_pred.prior["course_finish_time_multiplier_std"].values.flatten()
course_finish_time_multiplier_raw_prior = prior_pred.prior["course_finish_time_multiplier_raw"].values

# Compute variance of raw parameters across ALL courses (for each MCMC draw)
# Shape: (chains, draws, n_courses) -> compute std across courses -> (chains*draws,)
course_finish_time_multiplier_raw_var = course_finish_time_multiplier_raw_prior.std(axis=-1).flatten()

# Scatter plot: Hyperprior σ vs. Realized variance of raw parameters
fig, ax = plt.subplots(1, 1, figsize=(10, 10))

ax.scatter(course_finish_time_multiplier_std_prior_samples, course_finish_time_multiplier_raw_var,
          alpha=0.3, s=10, color='#4a4a4a')

# Add reference lines
ax.axhline(y=1.0, color='red', linestyle='--', linewidth=2, alpha=0.5, 
          label='Expected variance (σ=1 for non-centered)')
ax.set_xlabel('course_finish_time_multiplier_std (hyperprior σ)', fontsize=12, fontweight='bold')
ax.set_ylabel('Std(course_finish_time_multiplier_raw) across all courses', fontsize=12, fontweight='bold')
ax.set_title('Global Funnel Check: Hyperprior vs. Aggregate Course Variance', 
            fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## MCMC Inference

### Sampling

In [ ]:
# cache path
cache_file = f'{subset_dir}/tune{tune}_draws{draws}_accept{target_accept}.nc'

# Hyperparameters to monitor (finish time model)
hyperparam_vars = [
    'pace_marathon', 
    'pace_distance_effect', 
    'course_finish_time_multiplier_std', 
    'finish_time_noise'
]

# check if cached trace exists
if os.path.exists(cache_file):
    print(f"Loading cached trace from {cache_file}")
    trace = az.from_netcdf(cache_file)
    print(az.summary(trace, var_names=hyperparam_vars))
        
else:
    print(f"Configuration: tune={tune}, draws={draws}, target_accept={target_accept}")
    print(f"K-core subset: max_entities={max_entities}")
    
    # Send start notification
    notify_mcmc_start(
        model_name="Model 1",
        n_results=n_observations,
        draws=draws,
        target_accept=target_accept
    )
    start_time = time.time()
    
    try:
        with model:
            trace = pm.sample(
                draws=draws,  
                tune=tune,   
                chains=4,
                cores=4,
                target_accept=target_accept,  
                random_seed=42,
                return_inferencedata=True,
                idata_kwargs={"log_likelihood": False},
                init="jitter+adapt_diag",
                progressbar=True,
                compute_convergence_checks=False
            )
        
        elapsed_time = time.time() - start_time
        
        # Print summary before saving
        print(az.summary(trace, var_names=hyperparam_vars))
        
        # Save trace to cache
        print(f"Saving trace to {cache_file}...")
        trace.to_netcdf(cache_file)
        
        # Compute diagnostics for notification
        effective_draws = trace.posterior.dims['draw']
        divergences = trace.sample_stats.diverging.sum().values
        
        # Send success notification
        notify_mcmc_complete(
            model_name="Model 1",
            elapsed_time=elapsed_time,
            n_results=n_observations,
            effective_draws=effective_draws,
            divergences=divergences,
            n_chains=4
        )
        
        print(f"MCMC completed in {elapsed_time/60:.2f} minutes")
        
    except Exception as e:
        elapsed_time = time.time() - start_time
        
        # Send error notification
        notify_mcmc_error(
            model_name="Model 1",
            error_msg=str(e),
            elapsed_time=elapsed_time
        )
        
        print(f"❌ MCMC failed after {elapsed_time/60:.2f} minutes")
        raise


### Traceplot

In [ ]:
# traceplot for hyperparameters
az.plot_trace(trace, var_names=hyperparam_vars, compact=True, figsize=(12, 10))
plt.suptitle('MCMC Traces: Course-Level Difficulty Model Hyperparameters', fontsize=14, fontweight='bold')
plt.tight_layout()

# save traceplot
traceplot_file = f'{subset_dir}/tune{tune}_draws{draws}_accept{target_accept}_traceplot.png'
plt.savefig(traceplot_file, dpi=300, bbox_inches='tight')
print(f"Saved traceplot to {traceplot_file}")

plt.show()

# traceplot for sample of course difficulty parameters
course_coords = list(trace.posterior.coords['course'].values)
sample_course_ids = np.random.choice(course_coords, size=min(20, len(course_coords)), replace=False)

az.plot_trace(
    trace, 
    var_names=['course_finish_time_multiplier'],
    coords={'course': sample_course_ids},
    compact=True, 
    figsize=(12, 20)
)
plt.suptitle('MCMC Traces: Sample Course Finish Time Multiplier Parameters', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


### Posterior Diagnostics

Comprehensive diagnostic plots to check if the model converged

In [ ]:
plot_posterior_diagnostics(trace, hyperparam_vars, subset_dir, tune, draws, target_accept)

### Posterior Predictive Check

In [ ]:
with model:
    post_pred = pm.sample_posterior_predictive(
        trace, 
        random_seed=37,
        progressbar=True
    )

# add posterior predictive to trace
trace.extend(post_pred)

In [ ]:
# Extract posterior predictive samples
post_pred_times = post_pred.posterior_predictive["finish_times"].values
# Shape: (chains, draws, finishers) -> flatten to (samples, finishers)
post_pred_times_flat = post_pred_times.reshape(-1, post_pred_times.shape[-1])

# Gender label mapping for display
gender_display_names = {'M': 'Male', 'F': 'Female'}

# Use FULL dataset for observed distributions (not model_data subset)
# Extract data from results DataFrame
full_observed_times = results['time_ms'].values / 60000  # Convert to minutes
full_gender_indices = results['gender'].map(gender_to_idx).values
full_race_distances = results['distance_miles'].values

# Also extract K-CORE subset data for comparison
kcore_observed_times = observed_times  # This is from model_data (k-core subset)
kcore_gender_indices = gender_indices
kcore_race_distances = race_distances

# Create separate figure for each gender
for gender_idx, gender in enumerate(unique_genders):
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    axes = axes.flatten()
    
    # Masks for full dataset
    gender_mask = full_gender_indices == gender_idx
    # Masks for k-core subset
    kcore_gender_mask = kcore_gender_indices == gender_idx
    
    for idx, distance_tuple in enumerate(standard_distances):
        dist_value, name, tolerance = distance_tuple  # Explicit unpacking with correct order
        
        # Filter observed data for this distance and gender (from FULL dataset)
        obs_mask = np.abs(full_race_distances - dist_value) < tolerance
        combined_mask = obs_mask & gender_mask
        
        if not combined_mask.any():
            axes[idx].axis('off')
            axes[idx].set_title(f'{name}\nNo data', fontsize=9)
            continue
        
        # Observed times from FULL dataset
        obs_times_binned = full_observed_times[combined_mask]
        
        # Observed times from K-CORE subset
        kcore_obs_mask = np.abs(kcore_race_distances - dist_value) < tolerance
        kcore_combined_mask = kcore_obs_mask & kcore_gender_mask
        kcore_obs_times_binned = kcore_observed_times[kcore_combined_mask]
        
        # Predicted times (from model_data subset - these are still the posterior predictions)
        # We need to filter the posterior predictions to match the model_data indices
        model_obs_mask = np.abs(race_distances - dist_value) < tolerance
        model_gender_mask = gender_indices == gender_idx
        model_combined_mask = model_obs_mask & model_gender_mask
        
        if not model_combined_mask.any():
            # No model predictions for this distance/gender
            pred_times_flat = np.array([])
        else:
            pred_times_binned = post_pred_times_flat[:, model_combined_mask]
            pred_times_flat = pred_times_binned.flatten()
        
        # Calculate 99th percentile for x-axis limit
        if len(pred_times_flat) > 0:
            combined_data = np.concatenate([obs_times_binned, pred_times_flat, kcore_obs_times_binned])
        else:
            combined_data = np.concatenate([obs_times_binned, kcore_obs_times_binned])
        xlim_max = np.percentile(combined_data, 99)
        
        # Plot with automatic binning using step histograms
        # Only include labels for first subplot to create single legend
        if idx == 0:
            axes[idx].hist(obs_times_binned, bins='auto', histtype='step', linewidth=2,
                          label=f'Observed (All, n={len(obs_times_binned):,})', 
                          density=True, color='steelblue')
            axes[idx].hist(kcore_obs_times_binned, bins='auto', histtype='step', linewidth=2,
                          label=f'Observed (K-Core, n={len(kcore_obs_times_binned):,})', 
                          density=True, color='green')
            # Plot predicted distribution (from model)
            if len(pred_times_flat) > 0:
                axes[idx].hist(pred_times_flat, bins='auto', histtype='step', linewidth=2,
                              label='Predicted', 
                              density=True, color='orange')
        else:
            axes[idx].hist(obs_times_binned, bins='auto', histtype='step', linewidth=2,
                          density=True, color='steelblue')
            axes[idx].hist(kcore_obs_times_binned, bins='auto', histtype='step', linewidth=2,
                          density=True, color='green')
            if len(pred_times_flat) > 0:
                axes[idx].hist(pred_times_flat, bins='auto', histtype='step', linewidth=2,
                              density=True, color='orange')
        
        axes[idx].set_xlabel('Time (min)', fontsize=9)
        axes[idx].set_ylabel('Density', fontsize=9)
        axes[idx].set_title(f'{name} ({dist_value:.1f}mi)\nAll: {len(obs_times_binned):,} | K-Core: {len(kcore_obs_times_binned):,}', 
                            fontsize=9)
        axes[idx].tick_params(labelsize=8)
        axes[idx].set_xlim(0, xlim_max)
    
    # Create single horizontal legend below the title
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 0.98), 
              ncol=3, fontsize=10, frameon=False)
    
    fig.suptitle(f'Posterior Predictive Check: {gender_display_names[gender]}', 
                 fontsize=14, fontweight='bold', y=1.00)
    plt.tight_layout(rect=[0, 0, 1, 0.96])  # Leave space for legend
    plt.show()


### Parameter Visualizations

In [ ]:
# Pace-Distance Curves by Gender
# Extract posterior samples for pace_marathon and pace_distance_effect
pace_marathon_samples = trace.posterior["pace_marathon"].values  # Shape: (chains, draws, n_genders)
pace_distance_effect_samples = trace.posterior["pace_distance_effect"].values  # Shape: (chains, draws, n_genders)

# Create distance range for visualization
distances = np.linspace(5, 100, 200)  # 5 to 100 miles

# Plot both genders on the same figure
fig, ax = plt.subplots(1, 1, figsize=(14, 8))

# Define colors for each gender
gender_colors = {'M': 'steelblue', 'F': 'coral'}
gender_labels = {'M': 'Male', 'F': 'Female'}

# First pass: calculate all median curves to determine y-limits
all_pace_medians = {}
for gender_idx, gender in enumerate(unique_genders):
    pace_marathon_flat = pace_marathon_samples[:, :, gender_idx].flatten()
    pace_distance_effect_flat = pace_distance_effect_samples[:, :, gender_idx].flatten()
    
    log_pace_curves = []
    for pace_base, dist_effect in zip(pace_marathon_flat, pace_distance_effect_flat):
        log_pace = pace_base + dist_effect * np.log(distances / reference_distance)
        log_pace_curves.append(np.exp(log_pace))
    
    log_pace_curves = np.array(log_pace_curves)
    all_pace_medians[gender] = {
        'median': np.percentile(log_pace_curves, 50, axis=0),
        'p5': np.percentile(log_pace_curves, 5, axis=0),
        'p25': np.percentile(log_pace_curves, 25, axis=0),
        'p75': np.percentile(log_pace_curves, 75, axis=0),
        'p95': np.percentile(log_pace_curves, 95, axis=0)
    }

# Determine y-limits from all medians
all_median_values = np.concatenate([all_pace_medians[g]['median'] for g in unique_genders])
y_min = all_median_values.min() - 0.5
y_max = all_median_values.max() + 1.0
ax.set_ylim(y_min, y_max)

# Second pass: plot curves and markers
for gender_idx, gender in enumerate(unique_genders):
    pace_data = all_pace_medians[gender]
    pace_median = pace_data['median']
    pace_5 = pace_data['p5']
    pace_25 = pace_data['p25']
    pace_75 = pace_data['p75']
    pace_95 = pace_data['p95']
    
    color = gender_colors[gender]
    
    # Fill between for uncertainty
    ax.fill_between(distances, pace_5, pace_95, alpha=0.15, color=color)
    ax.fill_between(distances, pace_25, pace_75, alpha=0.25, color=color)
    
    # Median curve
    ax.plot(distances, pace_median, color=color, linewidth=2.5)
    
    # Add markers and annotations for ALL standard distances
    for dist_value, name, _ in standard_distances:
        # Find closest index in distances array
        idx = np.argmin(np.abs(distances - dist_value))
        pace = pace_median[idx]
        
        # Plot marker
        ax.plot(dist_value, pace, 'o', color=color, markersize=6, zorder=5, alpha=0.8)
        
        # Label with pace value (offset vertically by gender to avoid overlap)
        y_offset = 0.15 if gender == 'M' else -0.4
        label_text = f"{pace:.1f}"
        ax.text(dist_value, pace + y_offset, label_text, fontsize=7, 
                ha='center', va='bottom' if gender == 'M' else 'top', 
                fontweight='bold', color=color)

# Add distance labels at the top
y_label_position = y_max - 0.2  # Fixed offset from top
for dist_value, name, _ in standard_distances:
    ax.axvline(x=dist_value, color='gray', linestyle=':', alpha=0.3, linewidth=1)
    ax.text(dist_value, y_label_position, name, fontsize=8, 
            ha='center', va='top', rotation=0, alpha=0.6)

ax.set_xlabel('Distance (miles)', fontsize=13, fontweight='bold')
ax.set_ylabel('Pace (min/mile)', fontsize=13, fontweight='bold')
ax.set_title('Learned Pace-Distance Curves by Gender (Population Average)', 
             fontsize=15, fontweight='bold', pad=20)
ax.grid(alpha=0.3, linestyle='--')
ax.set_xlim(5, 100)

plt.tight_layout()
plt.show()


In [ ]:
# Forest plot of course difficulty (ordered by median)
# Extract course finish time multiplier values and compute medians for sorting
course_finish_time_multiplier_samples = trace.posterior["course_finish_time_multiplier"].values  # Shape: (chains, draws, n_courses)
course_names = list(trace.posterior.coords['course'].values)

# Calculate median for each course
medians = np.median(course_finish_time_multiplier_samples.reshape(-1, len(course_names)), axis=0)

# Create sorted indices (most difficult = highest positive value)
sorted_indices = np.argsort(medians)[::-1]  # Descending order

# Show top 20 hardest and bottom 30 easiest courses
n_show = 20
top_indices = sorted_indices[:n_show]
bottom_indices = sorted_indices[-n_show:]
selected_indices = np.concatenate([top_indices, bottom_indices])
selected_courses = [course_names[i] for i in selected_indices]

# Create forest plot
fig = az.plot_forest(
    trace,
    var_names=['course_finish_time_multiplier'],
    coords={'course': selected_courses},
    combined=True,
    figsize=(10, 12),
    ess=False,
    r_hat=False
)

# Add zero reference line
plt.axvline(x=0, linestyle='--', linewidth=1.5, alpha=0.5, label='Average Difficulty')
plt.xlabel('Course Difficulty (log pace adjustment)', fontsize=12, fontweight='bold')
plt.title(f'Course Difficulty Rankings\nTop {n_show} Hardest & Bottom {n_show} Easiest', 
          fontsize=14, fontweight='bold', pad=15)
plt.grid(alpha=0.3, axis='x')
plt.tight_layout()
plt.show()


## MAP Inference

Extend inference to the full dataset using learned hyperparameters from k-core MCMC.

The k-core subsetting approach provides high-quality estimates of **population-level hyperparameters** by focusing on densely-connected runners and courses. However, many courses appear only in the sparse closure (runners or courses with few observations). 

We use a **two-stage strategy**:
1. **MCMC on k-core**: Learn hyperparameters from dense subgraph (already completed above)
2. **MAP on full dataset**: Fix hyperparameters and estimate only course-level parameters for all courses

This approach is efficient because:
- Hyperparameters generalize from dense core to sparse closure
- Course multipliers are entity-specific adjustments (don't need full posterior)
- MAP optimization is fast (seconds/minutes vs. hours for full MCMC)


### Data Preparation

Extract fixed hyperparameters from k-core MCMC and prepare the full dataset for MAP estimation.

In [ ]:
# Extract posterior medians from k-core MCMC trace
# These hyperparameters will be FIXED in the MAP model
pace_marathon_fixed = trace.posterior["pace_marathon"].median(dim=["chain", "draw"]).values
pace_distance_effect_fixed = trace.posterior["pace_distance_effect"].median(dim=["chain", "draw"]).values
course_finish_time_multiplier_std_fixed = float(
    trace.posterior["course_finish_time_multiplier_std"].median(dim=["chain", "draw"]).values
)
finish_time_noise_fixed = trace.posterior["finish_time_noise"].median(dim=["chain", "draw"]).values

print("Fixed hyperparameters from k-core MCMC:")
print("=" * 60)
print(f"Finish Time Model:")
print(f"  pace_marathon (M, F):                    [{pace_marathon_fixed[0]:.4f}, {pace_marathon_fixed[1]:.4f}]")
print(f"  pace_distance_effect (M, F):             [{pace_distance_effect_fixed[0]:.4f}, {pace_distance_effect_fixed[1]:.4f}]")
print(f"  course_finish_time_multiplier_std:       {course_finish_time_multiplier_std_fixed:.4f}")
print(f"  finish_time_noise (M, F):                [{finish_time_noise_fixed[0]:.4f}, {finish_time_noise_fixed[1]:.4f}]")
print("=" * 60)


In [ ]:
# Prepare full dataset (non-subsetted results)
# Use the original results DataFrame which was already filtered for DNFs
# but NOT subsetted to k-core entities

# Create course index mapping for FULL dataset
unique_courses_full = results['name'].unique()
course_to_idx_full = {course: idx for idx, course in enumerate(unique_courses_full)}
results['course_idx_full'] = results['name'].map(course_to_idx_full)

# Create gender index mapping for FULL dataset (same as k-core)
results['gender_idx'] = results['gender'].map(gender_to_idx)

# Extract arrays for MAP model
n_courses_full = len(unique_courses_full)
n_observations_full = len(results)

course_indices_full = results['course_idx_full'].values
gender_indices_full = results['gender_idx'].values
race_distances_full = results['distance_miles'].values
observed_times_full = results['time_ms'].values / 60000  # Convert to minutes

print("\nDataset comparison:")
print("=" * 60)
print(f"K-core subset (MCMC):")
print(f"  Observations: {n_observations:,}")
print(f"  Courses:      {n_courses:,}")
print(f"\nFull dataset (MAP):")
print(f"  Observations: {n_observations_full:,}")
print(f"  Courses:      {n_courses_full:,}")
print(f"\nClosure (new entities):")
print(f"  Observations: {n_observations_full - n_observations:,} (+{100*(n_observations_full - n_observations)/n_observations:.1f}%)")
print(f"  Courses:      {n_courses_full - n_courses:,} (+{100*(n_courses_full - n_courses)/n_courses:.1f}%)")
print("=" * 60)

In [ ]:
# Verify k-core flags exist in model_data
# These flags were created by subset_kcore_data() function
print("K-core subsetting flags:")
print("=" * 60)
print(f"'in_kcore' column exists:   {'in_kcore' in model_data.columns}")
print(f"'in_closure' column exists: {'in_closure' in model_data.columns}")

if 'in_kcore' in model_data.columns and 'in_closure' in model_data.columns:
    print(f"\nK-core composition in MCMC subset:")
    print(f"  K-core results:   {model_data['in_kcore'].sum():,} ({100*model_data['in_kcore'].mean():.1f}%)")
    print(f"  Closure results:  {model_data['in_closure'].sum():,} ({100*model_data['in_closure'].mean():.1f}%)")
    print(f"\nNote: K-core means BOTH runner AND course are in dense core")
    print(f"      Closure means at least ONE entity is from sparse periphery")
print("=" * 60)

### Specification

Build a PyMC model with the same structure as the MCMC model, but with hyperparameters fixed to their posterior medians. Only course-level parameters will be estimated.

In [ ]:
# Create coordinate system for full dataset
coords_full = {
    'course': unique_courses_full,
    'gender': unique_genders,
    'finishers': range(n_observations_full),
}

with pm.Model(coords=coords_full) as model_map:
    
    # --- FIXED hyperparameters (from k-core MCMC) ---
    # Use pm.Data() to prevent these from being optimized
    pace_marathon = pm.Data('pace_marathon', pace_marathon_fixed, dims='gender')
    pace_distance_effect = pm.Data('pace_distance_effect', pace_distance_effect_fixed, dims='gender')
    course_finish_time_multiplier_std = pm.Data('course_finish_time_multiplier_std', course_finish_time_multiplier_std_fixed)
    finish_time_noise = pm.Data('finish_time_noise', finish_time_noise_fixed, dims='gender')
    
    # --- FREE parameters: Course finish time multipliers (non-centered parameterization) ---
    # These are the ONLY parameters that will be estimated by MAP
    course_finish_time_multiplier_raw = pm.Normal(
        'course_finish_time_multiplier_raw', 
        mu=0, 
        sigma=1, 
        dims='course'
    )
    course_finish_time_multiplier = pm.Deterministic(
        'course_finish_time_multiplier', 
        course_finish_time_multiplier_std * course_finish_time_multiplier_raw, 
        dims='course'
    )
    
    # --- Likelihood (same structure as MCMC model) ---
    log_distance_ratio = pm.math.log(race_distances_full / reference_distance)
    
    expected_log_pace = (
        pace_marathon[gender_indices_full] +
        pace_distance_effect[gender_indices_full] * log_distance_ratio +
        course_finish_time_multiplier[course_indices_full]
    )
    
    expected_pace = pm.Deterministic('expected_pace', pm.math.exp(expected_log_pace), dims='finishers')
    expected_time = expected_pace * race_distances_full
    
    pm.Normal(
        'finish_times', 
        mu=expected_time, 
        sigma=finish_time_noise[gender_indices_full] * race_distances_full, 
        observed=observed_times_full,
        dims='finishers'
    )

print("\nMAP Model Summary:")
print("=" * 60)
print(f"Total courses:        {n_courses_full:,}")
print(f"  K-core courses:     {n_courses:,}")
print(f"  Closure courses:    {n_courses_full - n_courses:,}")
print(f"\nFixed hyperparameters (from k-core MCMC):")
print(f"  pace_marathon, pace_distance_effect, course_finish_time_multiplier_std, finish_time_noise")
print(f"\nFree parameters (to be estimated):")
print(f"  course_finish_time_multiplier_raw (N = {n_courses_full:,})")

# Visualize model structure showing fixed vs free parameters
graph = pm.model_to_graphviz(model_map)
graph

### Estimation

Run fast optimization to find the mode of the posterior distribution (Maximum A Posteriori estimate). Unlike MCMC sampling, this produces point estimates only, but completes in seconds/minutes rather than hours.

In [ ]:
print("🚀 Running MAP estimation on full dataset...")
print(f"   Fixed hyperparameters from k-core MCMC")
print(f"   Estimating course_finish_time_multiplier for {n_courses_full:,} courses")
print()

start_time = time.time()

with model_map:
    map_estimate = pm.find_MAP(
        method='L-BFGS-B',
        progressbar=True
    )

elapsed_time = time.time() - start_time
print(f"\n✅ MAP estimation completed in {elapsed_time:.2f} seconds")

# Convert MAP point estimate to InferenceData format for consistency
# Create a single-draw posterior (shape: [1 chain, 1 draw, n_courses])
map_trace = az.from_dict(
    posterior={
        'course_finish_time_multiplier': map_estimate['course_finish_time_multiplier'][np.newaxis, np.newaxis, :],
        'course_finish_time_multiplier_raw': map_estimate['course_finish_time_multiplier_raw'][np.newaxis, np.newaxis, :]
    },
    coords={'course': unique_courses_full},
    dims={
        'course_finish_time_multiplier': ['course'],
        'course_finish_time_multiplier_raw': ['course']
    }
)

In [ ]:
# Extract MAP estimates and display summary statistics
map_course_finish_time_multiplier = map_trace.posterior['course_finish_time_multiplier'].values[0, 0, :]

### K-Core Validation

Validate MAP estimates by comparing them to MCMC posteriors for courses in the k-core subset. This checks whether the MAP optimization produces reasonable point estimates that align with the full Bayesian posterior.

In [ ]:
# Extract course finish time multipliers from both traces
# ALL courses in MCMC trace (both k-core and closure added up to max_entities=6000)
all_mcmc_courses = list(trace.posterior.coords['course'].values)

# MAP estimates for all courses in MCMC trace
map_course_names_full = list(map_trace.posterior.coords['course'].values)
mcmc_indices_in_map = [map_course_names_full.index(c) for c in all_mcmc_courses]
map_all_estimates = map_trace.posterior['course_finish_time_multiplier'].values[0, 0, mcmc_indices_in_map]

# Get MCMC posteriors for all courses
mcmc_medians = trace.posterior['course_finish_time_multiplier'].median(dim=['chain', 'draw']).values
mcmc_q05 = trace.posterior['course_finish_time_multiplier'].quantile(0.05, dim=['chain', 'draw']).values
mcmc_q95 = trace.posterior['course_finish_time_multiplier'].quantile(0.95, dim=['chain', 'draw']).values
mcmc_q25 = trace.posterior['course_finish_time_multiplier'].quantile(0.25, dim=['chain', 'draw']).values
mcmc_q75 = trace.posterior['course_finish_time_multiplier'].quantile(0.75, dim=['chain', 'draw']).values

# CORRECTED: Identify k-core vs closure-only courses from model_data
# K-core = dense core (152 courses)
# Closure-only = added entities not in k-core (288 courses)
kcore_course_names_from_data = model_data[model_data['in_kcore']]['name'].unique().tolist()
closure_course_names_from_data = model_data[model_data['in_closure']]['name'].unique().tolist()

# Separate MCMC courses: k-core (in k-core) vs closure-only (not in k-core)
is_kcore_course = np.array([name in kcore_course_names_from_data for name in all_mcmc_courses])
is_closure_only_course = ~is_kcore_course  # Closure entities NOT in k-core

map_kcore_estimates = map_all_estimates[is_kcore_course]
map_closure_only_estimates = map_all_estimates[is_closure_only_course]

# K-Core validation metrics (dense core)
kcore_indices = np.where(is_kcore_course)[0]
kcore_correlation = np.corrcoef(map_kcore_estimates, mcmc_medians[kcore_indices])[0, 1]
kcore_mae = np.mean(np.abs(map_kcore_estimates - mcmc_medians[kcore_indices]))
kcore_rmse = np.sqrt(np.mean((map_kcore_estimates - mcmc_medians[kcore_indices])**2))
kcore_within_90 = np.sum((map_kcore_estimates >= mcmc_q05[kcore_indices]) & 
                          (map_kcore_estimates <= mcmc_q95[kcore_indices]))
kcore_within_50 = np.sum((map_kcore_estimates >= mcmc_q25[kcore_indices]) & 
                          (map_kcore_estimates <= mcmc_q75[kcore_indices]))

# Closure-Only validation metrics (added entities not in k-core)
closure_only_indices = np.where(is_closure_only_course)[0]
closure_only_correlation = np.corrcoef(map_closure_only_estimates, mcmc_medians[closure_only_indices])[0, 1]
closure_only_mae = np.mean(np.abs(map_closure_only_estimates - mcmc_medians[closure_only_indices]))
closure_only_rmse = np.sqrt(np.mean((map_closure_only_estimates - mcmc_medians[closure_only_indices])**2))
closure_only_within_90 = np.sum((map_closure_only_estimates >= mcmc_q05[closure_only_indices]) & 
                                 (map_closure_only_estimates <= mcmc_q95[closure_only_indices]))
closure_only_within_50 = np.sum((map_closure_only_estimates >= mcmc_q25[closure_only_indices]) & 
                                 (map_closure_only_estimates <= mcmc_q75[closure_only_indices]))

print("\n" + "=" * 80)
print("MAP vs MCMC Validation (Finish Time Multipliers)")
print("=" * 80)
print(f"\nK-Core Courses (Dense Core, n={len(map_kcore_estimates):,}):")
print(f"  Correlation (r):        {kcore_correlation:.4f}")
print(f"  MAE:                    {kcore_mae:.4f}")
print(f"  RMSE:                   {kcore_rmse:.4f}")
print(f"  Within 90% CI:          {kcore_within_90}/{len(map_kcore_estimates)} ({100*kcore_within_90/len(map_kcore_estimates):.1f}%)")
print(f"  Within 50% CI:          {kcore_within_50}/{len(map_kcore_estimates)} ({100*kcore_within_50/len(map_kcore_estimates):.1f}%)")

print(f"\nClosure-Only Courses (Added Entities, n={len(map_closure_only_estimates):,}):")
print(f"  Correlation (r):        {closure_only_correlation:.4f}")
print(f"  MAE:                    {closure_only_mae:.4f}")
print(f"  RMSE:                   {closure_only_rmse:.4f}")
print(f"  Within 90% CI:          {closure_only_within_90}/{len(map_closure_only_estimates)} ({100*closure_only_within_90/len(map_closure_only_estimates):.1f}%)")
print(f"  Within 50% CI:          {closure_only_within_50}/{len(map_closure_only_estimates)} ({100*closure_only_within_50/len(map_closure_only_estimates):.1f}%)")
print("=" * 80)


In [ ]:
# Scatter plot: MAP vs MCMC with credible intervals - COURSE FINISH TIME MULTIPLIER
# K-core (dense core) vs Closure-only (added entities) courses

fig, ax = plt.subplots(1, 1, figsize=(14, 14))

# K-CORE COURSES: Dense core with credible intervals
mcmc_kcore_medians = mcmc_medians[is_kcore_course]
kcore_q25 = mcmc_q25[is_kcore_course]
kcore_q75 = mcmc_q75[is_kcore_course]
kcore_q05 = mcmc_q05[is_kcore_course]
kcore_q95 = mcmc_q95[is_kcore_course]

ax.errorbar(
    map_kcore_estimates, mcmc_kcore_medians,
    xerr=None,
    yerr=[mcmc_kcore_medians - kcore_q25, kcore_q75 - mcmc_kcore_medians],
    fmt='o', alpha=0.4, color='steelblue', markersize=4,
    elinewidth=1.5, capsize=0, label='K-Core 50% CI'
)

ax.errorbar(
    map_kcore_estimates, mcmc_kcore_medians,
    xerr=None,
    yerr=[mcmc_kcore_medians - kcore_q05, kcore_q95 - mcmc_kcore_medians],
    fmt='o', alpha=0.2, color='steelblue', markersize=4,
    elinewidth=0.8, capsize=0, label='K-Core 90% CI'
)

ax.scatter(map_kcore_estimates, mcmc_kcore_medians, s=30, color='steelblue', 
          alpha=0.6, zorder=5, label=f'K-Core Courses (n={sum(is_kcore_course):,})')

# CLOSURE-ONLY COURSES: Added entities with credible intervals
mcmc_closure_only_medians = mcmc_medians[is_closure_only_course]
closure_only_q25 = mcmc_q25[is_closure_only_course]
closure_only_q75 = mcmc_q75[is_closure_only_course]
closure_only_q05 = mcmc_q05[is_closure_only_course]
closure_only_q95 = mcmc_q95[is_closure_only_course]

ax.errorbar(
    map_closure_only_estimates, mcmc_closure_only_medians,
    xerr=None,
    yerr=[mcmc_closure_only_medians - closure_only_q25, closure_only_q75 - mcmc_closure_only_medians],
    fmt='^', alpha=0.3, color='limegreen', markersize=3,
    elinewidth=1.0, capsize=0, label='Closure-Only 50% CI'
)

ax.errorbar(
    map_closure_only_estimates, mcmc_closure_only_medians,
    xerr=None,
    yerr=[mcmc_closure_only_medians - closure_only_q05, closure_only_q95 - mcmc_closure_only_medians],
    fmt='^', alpha=0.15, color='limegreen', markersize=3,
    elinewidth=0.6, capsize=0, label='Closure-Only 90% CI'
)

ax.scatter(map_closure_only_estimates, mcmc_closure_only_medians, s=20, color='limegreen', 
          alpha=0.5, zorder=4, marker='^', label=f'Closure-Only Courses (n={sum(is_closure_only_course):,})')

# Add y=x reference line (perfect agreement)
lims = [
    min(map_all_estimates.min(), mcmc_medians.min()) - 0.1,
    max(map_all_estimates.max(), mcmc_medians.max()) + 0.1
]
ax.plot(lims, lims, 'r--', linewidth=2, alpha=0.7, label='Perfect Agreement (y=x)')

# Fit lines for each group
from scipy.stats import linregress
kcore_slope, kcore_intercept, _, _, _ = linregress(map_kcore_estimates, mcmc_kcore_medians)
kcore_fit_x = np.array([map_kcore_estimates.min(), map_kcore_estimates.max()])
kcore_fit_y = kcore_slope * kcore_fit_x + kcore_intercept
ax.plot(kcore_fit_x, kcore_fit_y, color='steelblue', linewidth=2, alpha=0.7,
       linestyle=':', label=f'K-Core Fit (slope={kcore_slope:.3f})')

closure_only_slope, closure_only_intercept, _, _, _ = linregress(map_closure_only_estimates, mcmc_closure_only_medians)
closure_only_fit_x = np.array([map_closure_only_estimates.min(), map_closure_only_estimates.max()])
closure_only_fit_y = closure_only_slope * closure_only_fit_x + closure_only_intercept
ax.plot(closure_only_fit_x, closure_only_fit_y, color='limegreen', linewidth=2, alpha=0.7,
       linestyle=':', label=f'Closure-Only Fit (slope={closure_only_slope:.3f})')

# Add correlation and MAE annotations
ax.text(0.05, 0.95, 
       f'K-Core:\n  r = {kcore_correlation:.4f}\n  MAE = {kcore_mae:.4f}\n  RMSE = {kcore_rmse:.4f}',
       transform=ax.transAxes, fontsize=11, verticalalignment='top',
       bbox=dict(boxstyle='round', facecolor='steelblue', alpha=0.4))

ax.text(0.95, 0.95, 
       f'Closure-Only:\n  r = {closure_only_correlation:.4f}\n  MAE = {closure_only_mae:.4f}\n  RMSE = {closure_only_rmse:.4f}',
       transform=ax.transAxes, fontsize=11, verticalalignment='top',
       horizontalalignment='right',
       bbox=dict(boxstyle='round', facecolor='limegreen', alpha=0.4))

ax.set_xlabel('MAP Estimate (Full Dataset)', fontsize=12, fontweight='bold')
ax.set_ylabel('MCMC Posterior Median (K-Core Subset)', fontsize=12, fontweight='bold')
ax.set_title('Validation: MAP vs MCMC Course Finish Time Multipliers\n(K-Core Dense Core vs Closure-Only Added Entities)', 
            fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=9, ncol=2)
ax.grid(alpha=0.3)
ax.set_xlim(lims)
ax.set_ylim(lims)

plt.tight_layout()
plt.show()

# Identify outliers (MAP outside 90% credible interval) for k-core courses
kcore_outliers_mask = (map_kcore_estimates < kcore_q05) | (map_kcore_estimates > kcore_q95)
if kcore_outliers_mask.any():
    print(f"\n⚠️  K-Core Outliers ({kcore_outliers_mask.sum()} courses where MAP is outside 90% CI):")
    print("=" * 80)
    kcore_course_names = [all_mcmc_courses[i] for i in range(len(all_mcmc_courses)) if is_kcore_course[i]]
    outlier_indices = np.where(kcore_outliers_mask)[0]
    for idx in outlier_indices[:10]:  # Show first 10
        course_name = kcore_course_names[idx]
        map_val = map_kcore_estimates[idx]
        mcmc_val = mcmc_kcore_medians[idx]
        ci_lower = kcore_q05[idx]
        ci_upper = kcore_q95[idx]
        print(f"  {course_name[:60]:<60} MAP: {map_val:+.3f}  MCMC: {mcmc_val:+.3f} [{ci_lower:+.3f}, {ci_upper:+.3f}]")
    if kcore_outliers_mask.sum() > 10:
        print(f"  ... and {kcore_outliers_mask.sum() - 10} more")
else:
    print("\n✅ No k-core outliers: All MAP estimates within MCMC 90% credible intervals!")


In [ ]:
# MAP vs MCMC Validation: K-Core and Closure-Only Courses
# Compare MAP estimates to MCMC posteriors

from scipy import stats
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('MAP vs MCMC', fontsize=16, fontweight='bold', y=1.00)

# Left: K-core courses (dense core) - MAP vs MCMC % differences
ax = axes[0]

# Get MAP and MCMC estimates for k-core courses
kcore_indices = np.where(is_kcore_course)[0]
kcore_map = map_kcore_estimates
kcore_mcmc = mcmc_medians[kcore_indices]

# Compute percentage differences: 100 * (MAP - MCMC) / MCMC
# Positive = MAP overestimates difficulty, Negative = MAP underestimates
kcore_differences = 100 * (kcore_map - kcore_mcmc) / kcore_mcmc

# Calculate percentile-based x-axis limits to exclude outliers (keep 95% of data)
kcore_p2_5 = np.percentile(kcore_differences, 2.5)
kcore_p97_5 = np.percentile(kcore_differences, 97.5)
kcore_xlim = max(abs(kcore_p2_5), abs(kcore_p97_5))  # Symmetric around zero

# Create bins within the display range
kcore_bins = np.linspace(-kcore_xlim, kcore_xlim, 50)
ax.hist(kcore_differences, bins=kcore_bins, alpha=0.7, histtype='step', linewidth=2,
       color='steelblue', edgecolor='navy')

# Add vertical line at zero (perfect agreement)
ax.axvline(x=0, linestyle='--', color='red', linewidth=2, 
          label='Perfect Agreement', alpha=0.7)

# Add summary statistics
kcore_mean_diff = kcore_differences.mean()
kcore_std_diff = kcore_differences.std()
kcore_median_diff = np.median(kcore_differences)
kcore_mae = np.mean(np.abs(kcore_differences))

stats_text = f"K-Core Stats (n={len(kcore_differences)}):\n  Mean: {kcore_mean_diff:+.2f}%\n  Median: {kcore_median_diff:+.2f}%\n  MAE: {kcore_mae:.2f}%\n  Std: {kcore_std_diff:.2f}%"
ax.text(0.98, 0.98, stats_text, transform=ax.transAxes, 
       fontsize=10, verticalalignment='top', horizontalalignment='right',
       bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.6))

ax.set_xlabel('% Difference', fontsize=12, fontweight='bold')
ax.set_ylabel('Frequency', fontsize=12, fontweight='bold')
ax.set_title('K-Core', fontsize=13, fontweight='bold')
ax.set_xlim(-kcore_xlim, kcore_xlim)  # Centered on zero
ax.legend(fontsize=11)
ax.grid(alpha=0.3)

# Right: Closure-only courses (added entities) - MAP vs MCMC % differences
ax = axes[1]

# Get MAP and MCMC estimates for closure-only courses
closure_only_indices = np.where(is_closure_only_course)[0]
closure_only_map = map_closure_only_estimates
closure_only_mcmc = mcmc_medians[closure_only_indices]

# Compute percentage differences: 100 * (MAP - MCMC) / MCMC
closure_only_differences = 100 * (closure_only_map - closure_only_mcmc) / closure_only_mcmc

# Calculate percentile-based x-axis limits to exclude outliers (keep 95% of data)
closure_p2_5 = np.percentile(closure_only_differences, 2.5)
closure_p97_5 = np.percentile(closure_only_differences, 97.5)
closure_xlim = max(abs(closure_p2_5), abs(closure_p97_5))  # Symmetric around zero

# Create bins within the display range
closure_bins = np.linspace(-closure_xlim, closure_xlim, 50)
ax.hist(closure_only_differences, bins=closure_bins, alpha=0.7, histtype='step', linewidth=2,
       color='limegreen', edgecolor='darkgreen')

# Add vertical line at zero (perfect agreement)
ax.axvline(x=0, linestyle='--', color='red', linewidth=2, 
          label='Perfect Agreement', alpha=0.7)

# Add summary statistics
closure_mean_diff = closure_only_differences.mean()
closure_std_diff = closure_only_differences.std()
closure_median_diff = np.median(closure_only_differences)
closure_mae = np.mean(np.abs(closure_only_differences))

stats_text = f"Closure-Only Stats (n={len(closure_only_differences)}):\n  Mean: {closure_mean_diff:+.2f}%\n  Median: {closure_median_diff:+.2f}%\n  MAE: {closure_mae:.2f}%\n  Std: {closure_std_diff:.2f}%"
ax.text(0.98, 0.98, stats_text, transform=ax.transAxes, 
       fontsize=10, verticalalignment='top', horizontalalignment='right',
       bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.6))

ax.set_xlabel('% Difference', fontsize=12, fontweight='bold')
ax.set_ylabel('Frequency', fontsize=12, fontweight='bold')
ax.set_title('Closure-Only', fontsize=13, fontweight='bold')
ax.set_xlim(-closure_xlim, closure_xlim)  # Centered on zero
ax.legend(fontsize=11)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()


### Parameter Visualizations

Visualize course difficulty rankings and distributions using MAP estimates for the complete dataset (k-core + closure courses).

In [ ]:
# Forest plot: Top 50 Hardest and Bottom 50 Easiest Courses (Full Dataset)
# Sort all courses by difficulty (course_finish_time_multiplier)
all_course_names_map = list(map_trace.posterior.coords['course'].values)
sorted_indices = np.argsort(map_course_finish_time_multiplier)[::-1]  # Descending order

# Select top 50 hardest and bottom 50 easiest
n_show = 50
top_indices = sorted_indices[:n_show]
bottom_indices = sorted_indices[-n_show:]
selected_indices = np.concatenate([top_indices, bottom_indices])

# Create figure manually (arviz.plot_forest doesn't support markers by group)
fig, ax = plt.subplots(1, 1, figsize=(10, 20))

# Extract finish time multipliers and course names for selected courses
selected_multipliers = map_course_finish_time_multiplier[selected_indices]
selected_names = [all_course_names_map[i] for i in selected_indices]

# Check which courses are in MCMC trace vs MAP-only
# Three groups:
# 1. K-core: In MCMC dense core (from kcore_course_names_from_data)
# 2. Closure-only: In MCMC but not k-core (from all_mcmc_courses but not k-core)
# 3. MAP-only: NOT in MCMC at all (only in full dataset MAP)
kcore_set = set(kcore_course_names_from_data)
mcmc_set = set(all_mcmc_courses)  # All MCMC courses

selected_is_kcore = [all_course_names_map[i] in kcore_set for i in selected_indices]
selected_is_closure_only = [all_course_names_map[i] in mcmc_set and all_course_names_map[i] not in kcore_set for i in selected_indices]
selected_is_map_only = [all_course_names_map[i] not in mcmc_set for i in selected_indices]

# Get MCMC posteriors for courses that have them
selected_mcmc_medians = []
selected_mcmc_q05 = []
selected_mcmc_q95 = []
for course_name in selected_names:
    if course_name in all_mcmc_courses:
        idx = all_mcmc_courses.index(course_name)
        selected_mcmc_medians.append(mcmc_medians[idx])
        selected_mcmc_q05.append(mcmc_q05[idx])
        selected_mcmc_q95.append(mcmc_q95[idx])
    else:
        selected_mcmc_medians.append(None)
        selected_mcmc_q05.append(None)
        selected_mcmc_q95.append(None)

# Plot as horizontal lines with markers
y_positions = np.arange(len(selected_multipliers))

for i, (multiplier, is_kcore, is_closure, is_map_only) in enumerate(
    zip(selected_multipliers, selected_is_kcore, selected_is_closure_only, selected_is_map_only)):
    
    # Always plot orange square for MAP estimate
    ax.plot([multiplier], [y_positions[i]], marker='s', color='orange', 
           markersize=8, alpha=0.7, zorder=2, markeredgecolor='darkorange', markeredgewidth=0.5)
    
    # Plot MCMC posterior with credible interval if available
    if not is_map_only and selected_mcmc_medians[i] is not None:
        mcmc_median = selected_mcmc_medians[i]
        mcmc_q05_val = selected_mcmc_q05[i]
        mcmc_q95_val = selected_mcmc_q95[i]
        
        if is_kcore:
            # K-core: blue circle with credible interval
            ax.errorbar([mcmc_median], [y_positions[i]], 
                       xerr=[[mcmc_median - mcmc_q05_val], [mcmc_q95_val - mcmc_median]],
                       fmt='o', color='steelblue', markersize=5, alpha=0.9, 
                       elinewidth=1.5, capsize=3, zorder=3, 
                       markeredgecolor='navy', markeredgewidth=0.5)
        elif is_closure:
            # Closure-only: green triangle with credible interval
            ax.errorbar([mcmc_median], [y_positions[i]], 
                       xerr=[[mcmc_median - mcmc_q05_val], [mcmc_q95_val - mcmc_median]],
                       fmt='^', color='limegreen', markersize=5, alpha=0.9, 
                       elinewidth=1.5, capsize=3, zorder=3, 
                       markeredgecolor='darkgreen', markeredgewidth=0.5)

# Add zero reference line
ax.axvline(x=0, linestyle='--', color='red', linewidth=2, alpha=0.5, 
          label='Average Difficulty')

# Configure axes
ax.set_yticks(y_positions)
ax.set_yticklabels(selected_names, fontsize=8)
ax.set_xlabel('Course Difficulty (log pace adjustment)', fontsize=12, fontweight='bold')
ax.set_title(f'Course Difficulty Rankings (Full Dataset)\nTop {n_show} Hardest & Bottom {n_show} Easiest', 
            fontsize=14, fontweight='bold', pad=15)
ax.grid(alpha=0.3, axis='x')
ax.invert_yaxis()  # Hardest at top

# Add legend
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='s', color='w', markerfacecolor='orange', 
          markersize=8, markeredgecolor='darkorange', markeredgewidth=0.5,
          label=f'All Courses (MAP estimates) - {len(selected_multipliers)} shown'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='steelblue', 
          markersize=6, markeredgecolor='navy', markeredgewidth=0.5,
          label=f'K-Core Dense Core (MCMC + MAP) - {sum(selected_is_kcore)} shown'),
    Line2D([0], [0], marker='^', color='w', markerfacecolor='limegreen', 
          markersize=6, markeredgecolor='darkgreen', markeredgewidth=0.5,
          label=f'Closure-Only (MCMC + MAP) - {sum(selected_is_closure_only)} shown'),
    Line2D([0], [0], marker='s', color='w', markerfacecolor='orange', 
          markersize=8, markeredgecolor='darkorange', markeredgewidth=0.5,
          label=f'MAP-Only (no MCMC) - {sum(selected_is_map_only)} shown'),
    Line2D([0], [0], color='red', linestyle='--', linewidth=2, 
          label='Average Difficulty')
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=10)

plt.tight_layout()
plt.show()
